In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
from numpy.linalg import eigh, eigvalsh
import matplotlib.pyplot as plt
import pandas as pd

import pathlib
curr_dir = pathlib.Path(".")

import pyomo.environ as pyo
import cvxpy as cp

from core.neural_networks import *

In [ ]:
# # Choose M with underlying graph structure that admits exact SDP relaxation
# n = 7
# M = np.array([
#     [3, 3, 3, 2, 2, -3, -2],
#     [3, 2, -2, 0, 0, 0, 0],
#     [3, -2, 2, 0, 0, 0, 0],
#     [2, 0, 0, -2, -3, 0, 0],
#     [2, 0, 0, -3, 2, 0, 0],
#     [-3, 0, 0, 0, 0, -3, 0],
#     [-2, 0, 0, 0, 0, 0, 3],
# ])

# assert np.all(M == M.T)
# print(M)

# eigvals = eigvalsh(M)
# print(min(eigvals), max(eigvals)) # Should be sign indefinite

In [ ]:
# # Local Search Model (pyomo)

# def x(m):
#     return np.array([pyo.value(m.x[j]) for j in m.inds])

# model = pyo.ConcreteModel()

# # Sets
# model.inds = pyo.Set(initialize=np.arange(n))

# # Parameters
# model.M = pyo.Param(model.inds, model.inds, initialize={(i,j): M[i,j] for i in model.inds for j in model.inds})

# # Variables
# model.x = pyo.Var(model.inds, initialize={j: np.sqrt(b[j]) for j in model.inds})

# # Objective
# def obj(m):
#     return sum(m.x[i] * m.M[i,j] * m.x[j] for i in m.inds for j in m.inds)
# model.obj = pyo.Objective(rule=obj, sense=pyo.minimize)

# # Constraints
# def constr(m, j):
#     return m.x[j]**2 == b[j]
# model.constr = pyo.Constraint(model.inds, rule=constr)

# # Solve
# solver = pyo.SolverFactory('ipopt')
# solver.solve(model, tee=False)

# print("Locally Optimal Objective Function Value:")
# print(pyo.value(obj(model)))

# print("Locally Optimal Solution:")
# print(x(model))

In [ ]:
# # Mixed-Integer Model (pyomo)

# def x(m):
#     return np.array([pyo.value(m.x[j]) for j in m.inds])

# def u(m):
#     return np.array([pyo.value(m.u[j]) for j in m.inds])

# model = pyo.ConcreteModel()

# # Sets
# model.inds = pyo.Set(initialize=np.arange(n))

# # Parameters
# model.M = pyo.Param(model.inds, model.inds, initialize={(i,j): M[i,j] for i in model.inds for j in model.inds})

# # Variables
# model.x = pyo.Var(model.inds, initialize={j: 1.0 for j in model.inds})
# model.u = pyo.Var(model.inds, within=pyo.Binary, initialize={j: 1.0 for j in model.inds})

# # Objective
# def obj(m):
#     return sum(m.x[i] * m.M[i,j] * m.x[j] for i in m.inds for j in m.inds)
# model.obj = pyo.Objective(rule=obj, sense=pyo.minimize)

# # Constraints
# def constr(m, j):
#     return m.x[j] == np.sqrt(b[j]) * (2 * m.u[j] - 1)
# model.constr = pyo.Constraint(model.inds, rule=constr)

# # Sign constraint; x[1] == 1
# def sign_constr(m):
#     return m.x[0] == np.sqrt(b[0])
# model.sign_constr = pyo.Constraint(rule=sign_constr)

# # Solve
# solver = pyo.SolverFactory('gurobi')
# solver.solve(model, tee=True)

# print("Globally Optimal Objective Function Value:")
# print(pyo.value(obj(model)))

# print("Globally Optimal Solution:")
# print(x(model))

# print("Binary Variables:")
# print(u(model))

In [ ]:
# # SDP Model (cvxpy)

# # Variables
# X = cp.Variable((n, n), symmetric=True)

# # Objective
# objective = cp.Minimize(cp.trace(M @ X))

# # Constraints
# constraints = [X >> 0]
# for j in range(n):
#     constraints += [X[j,j] == b[j]]

# # Problem
# prob = cp.Problem(objective, constraints)
# result = prob.solve()

# print("Global Objective Function Lower Bound:")
# print(result)

# # Check rank
# tol = 1e-6
# eigvals, eigvecs = np.linalg.eigh(X.value)
# rank = np.sum(eigvals > tol)
# assert rank == 1.0

# # Recover optimal solution
# idx = np.argmax(eigvals)
# lambda1 = eigvals[idx]
# v = eigvecs[:, idx]
# x = np.sqrt(lambda1) * v

# print("Globally Optimal Objective Function Value:")
# print(result)

# print("Globally Optimal Solution:")
# print(x)

### Example Problem

$$
\begin{align}
\min~& (x - a)^2 + (y - b)^2 \\
\text{s.t.}~& xy \geq 1
\end{align}
$$

In [ ]:
data_dir = curr_dir / "training_data" / "QCQP_example"
data_dir.mkdir(exist_ok=True)
model_dir = curr_dir / "models" / "QCQP_example"
model_dir.mkdir(exist_ok=True)
results_dir = curr_dir / "results" / "QCQP_example"
results_dir.mkdir(exist_ok=True)
figures_dir = curr_dir / "figures"
figures_dir.mkdir(exist_ok=True)
logs_dir = curr_dir / "logs"
logs_dir.mkdir(exist_ok=True)

iter_dict = {}

In [ ]:
a, b = -2, 1

x = np.linspace(-5, 5, 1000)
xpos = x[x > 0]
xneg = x[x < 0]
ypos = 1 / xpos
yneg = 1 / xneg

plt.figure()
plt.scatter([a], [b], s=30, c="green", label="(a,b)")
plt.plot(xpos, ypos, c="k", label="xy = 1")
plt.plot(xneg, yneg, c="k")
plt.plot(x, 0*x, c="k", linewidth=0.3, linestyle="--")
plt.plot(0*x, x, c="k", linewidth=0.3, linestyle="--")
plt.axis([-5, 5, -5, 5])
plt.legend()
plt.show()

In [ ]:
# Local Search Model (pyomo)

model = pyo.ConcreteModel()

# Variables
x0, y0 = 3, 1.75
model.x = pyo.Var(initialize=x0)
model.y = pyo.Var(initialize=y0)

# Parameters
model.a = pyo.Param(initialize=a, mutable=True)
model.b = pyo.Param(initialize=b, mutable=True)

# Objective
def obj(m):
    return (m.x - m.a)**2 + (m.y - m.b)**2
model.obj = pyo.Objective(rule=obj, sense=pyo.minimize)

# Constraints
def constr(m):
    return m.x * m.y >= 1
model.constr = pyo.Constraint(rule=constr)

# Solve
solver = pyo.SolverFactory("ipopt")
solver.options["print_level"] = 8
solver.options["file_print_level"] = 8
solver.options["output_file"] = str(logs_dir / "ipopt_log.txt")
solver.options['max_iter'] = 100
solver.solve(model, tee=False)

print("Locally Optimal Objective Function Value:")
print(pyo.value(obj(model)))

print("Locally Optimal Solution:")
print(pyo.value(model.x), pyo.value(model.y))

In [ ]:
# Parse log file to get iterates
with open(logs_dir / "ipopt_log.txt", "r") as f:
    lines = f.readlines()
curr_x_lines = [line.strip() for line in lines if "curr_x[" in line]
x_iters = [float(line[-23:]) for line in curr_x_lines if "1]" in line]
y_iters = [float(line[-23:]) for line in curr_x_lines if "2]" in line]

In [ ]:
#del iter_dict[(-1, 1)]

In [ ]:
iter_dict[(x0, y0)] = (x_iters, y_iters)

In [ ]:
plt.figure(figsize=(5, 5))
plt.gca().set_aspect("equal")
plt.rcParams['font.size'] = 12

plt.scatter([a], [b], s=50, c="green", label=f"p = ({int(a)},{int(b)})")

for (x0, y0) in iter_dict.keys():
    x_iters, y_iters = iter_dict[(x0, y0)]
    plt.scatter([x0], [y0], s=30, c="red", zorder=3)
    plt.scatter([x_iters[-1]], [y_iters[-1]], s=50, c="blue", zorder=3)
    plt.plot(x_iters, y_iters, color="black", linestyle="--")

plt.scatter([100], [100], c="red", label="Initial Points")
plt.scatter([100], [100], c="blue", label="Local Minima")
plt.plot(xpos, ypos, c="k", linewidth=2.0)
plt.plot(xneg, yneg, c="k", linewidth=2.0)
plt.fill_between(xpos, 1/xpos, 5, color="lightgreen", alpha=0.5, label="Feasible")
plt.fill_between(xneg, -5, 1/xneg, color="lightgreen", alpha=0.5)

plt.plot(x, 0*x, c="k", linewidth=0.3, linestyle="--")
plt.plot(0*x, x, c="k", linewidth=0.3, linestyle="--")
plt.axis([-5, 5, -5, 5])
plt.legend(fontsize=10)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.savefig(figures_dir / "QCQP_example_convergence.pdf")
plt.show()

In [ ]:
def lss(a, b):
    model.x.value = 1.0
    model.y.value = 1.0
    model.a.value = a
    model.b.value = b
    solver.solve(model, tee=False)
    return pyo.value(obj(model))

In [ ]:
# SDP Model (cvxpy)

# Define matrices
def M0_func(a, b):
    return np.array([
        [a**2 + b**2, -a, -b],
        [-a, 1, 0],
        [-b, 0, 1],
    ])

M1 = np.array([
    [0, 0, 0],
    [0, 0, -0.5],
    [0, -0.5, 0],
])

# Parameters
M0 = cp.Parameter((3, 3))
M0.value = M0_func(a, b)

# Variables
X = cp.Variable((3, 3), symmetric=True)

# Objective
objective = cp.Minimize(cp.trace(M0 @ X))

# Constraints
constraints = [cp.trace(M1 @ X) <= -1, X[0,0] == 1, X >> 0]

# Problem
prob = cp.Problem(objective, constraints)
result = prob.solve()

print("Global Objective Function Lower Bound:")
print(result)

# Check rank
tol = 1e-6
eigvals, eigvecs = np.linalg.eigh(X.value)
rank = np.sum(eigvals > tol)
assert rank == 1.0

# Recover optimal solution
idx = np.argmax(eigvals)
lambda1 = eigvals[idx]
v = eigvecs[:, idx]
x = np.sqrt(lambda1) * v

print("Globally Optimal Objective Function Value:")
print(result)

print("Globally Optimal Solution:")
print(x)

In [ ]:
def v(a, b):
    M0.value = M0_func(a, b)
    result = prob.solve()
    # Check rank
    tol = 1e-6
    eigvals, eigvecs = np.linalg.eigh(X.value)
    rank = np.sum(eigvals > tol)
    try:
        assert rank == 1.0
    except:
        print(a, b)
    return result

In [ ]:
# Parameter ranges
a_vals = np.linspace(-5, 5, 50)
b_vals = np.linspace(-5, 5, 50)

# Make a mesh grid
A, B = np.meshgrid(a_vals, b_vals)

# Evaluate v(a, b) over the grid
V = np.zeros_like(A, dtype=float)
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        V[i, j] = v(A[i, j], B[i, j])
        break

In [ ]:
nn_model = FCNN(input_dim=2)
nn_save_path = model_dir / f"nn_QCQP_example_2d_5000samples.pth"
load_model(nn_model, nn_save_path)

# Get predictions over grid
coords = np.stack([A.ravel(), B.ravel()], axis=1)  # shape: (N, 2)
coords_torch = torch.tensor(coords, dtype=torch.float32)
with torch.no_grad():
    preds = nn_model(coords_torch).numpy()
V_hat = preds.reshape(A.shape)

In [ ]:
# Create the 3D plot
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(A, B, V, cmap='viridis')

# Label axes
ax.set_xlabel(r'$p_1$')
ax.set_ylabel(r'$p_2$')
ax.set_zlabel(r'$v(p)$')

plt.tight_layout()
plt.show()

In [ ]:
# Create the 3D plot
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(A, B, V_hat, cmap='viridis')

# Label axes
ax.set_xlabel(r'$p_1$')
ax.set_ylabel(r'$p_2$')
ax.set_zlabel(r'$\hat{v}(p)$')

# Adjust layout manually to prevent label cutoff
#fig.subplots_adjust(left=0.0, right=0.95, bottom=0.1, top=1.0)

# Save and show
plt.savefig(figures_dir / "3d_value_function_plot.pdf", bbox_inches='tight', pad_inches=0.3)
plt.show()

In [ ]:
X_arr = 10 * np.random.rand(5000, 2) - 5
y_arr = np.array([v(X_arr[i, 0], X_arr[i, 1]) for i in range(X_arr.shape[0])]).reshape(-1, 1)

In [ ]:
X_columns = ["a", "b"]
df_data = pd.DataFrame(columns=X_columns + ["Cost"])
df_data[X_columns] = X
df_data["Cost"] = y
df_data.to_csv(data_dir / "training_QCQP_example_2d_5000samples.csv")

In [ ]:
from core.neural_networks import *

In [ ]:
# NN

# Create training/testing data sets
X_train, y_train, X_test, y_test = create_train_test_datasets(X, y)
train_loader1 = create_batches(X_train, y_train, batch_size=250)
train_loader2 = create_batches(X_train, y_train, batch_size=50)
test_loader = create_batches(X_test, y_test, batch_size=100)

# Initialize model
nn_model = FCNN(input_dim=2)

# Train model

nn_model, nn_train_history_phase2, nn_test_history_phase2 = train_model(
    nn_model,
    train_loader2,
    test_loader,
    optimizer="Adam", 
    n_epochs=3000,
    learning_rate=0.01,
    weight_decay=1e-9,
    scheduler_step_size=250,
    scheduler_gamma=0.9,
    max_norm=100.0
)
nn_train_history = nn_train_history_phase2
nn_test_history = nn_test_history_phase2

nn_save_path = model_dir / f"nn_QCQP_example_2d_5000samples.pth"
SAVE = True
if SAVE:
    save_model(nn_model, nn_save_path)

In [ ]:
# -----------------------------
# Plotting
# -----------------------------

plt.title("NN Train / Test Error")
plt.plot(np.sqrt(np.array(nn_train_history))[10:], label="Train")
plt.plot(np.sqrt(np.array(nn_test_history))[10:], label="Test")
plt.legend()
#plt.yscale('log')
#plt.axis([0, None, 0, 10])
plt.show()

In [ ]:
# Create test data tensor
test_loader = create_batches(X_test, y_test, batch_size=1, shuffle=False)

# Get NN, ICNN predictions
print("Getting NN predictions...")
nn_pred_arr = pred(nn_model, test_loader)
print("Done.")

# Get local search solver OPF predictions
print("Getting local search predictions...")
recalculate = False
if recalculate:
    lss_pred_arr = np.array([lss(X_test[i,0], X_test[i,1]) for i in range(X_test.shape[0])])
print("Done.")

# Save results
df_results = pd.DataFrame(columns=X_columns + ["Cost", "LSS Pred", "NN Pred"])
df_results[X_columns] = X_test
df_results["Cost"] = y_test
df_results["LSS Pred"] = lss_pred_arr
df_results["NN Pred"] = nn_pred_arr

df_results.to_csv(results_dir / f"results_QCQP_example_2d_5000samples.csv")

# Calculate residuals
cost_arr = y_test.flatten()
nn_residuals = (nn_pred_arr - cost_arr)
lss_residuals = (lss_pred_arr - cost_arr)

In [ ]:
plt.figure(figsize=(8,5))
bins = np.linspace(-10, 100, 50)
plt.hist(lss_residuals, bins=bins, color="orange", alpha=0.8, label="Local Search")
plt.hist(nn_residuals, bins=bins, color="blue", alpha=0.5, label="NN")
plt.legend()
plt.xlabel("Cost Residual")
plt.show()

In [ ]:
print(f"LSS Error: {np.mean(np.abs(lss_residuals)):.2f} % Avg.  / {np.abs(lss_residuals).max():.2f} % Max.")
print(f"NN Error: {np.mean(np.abs(nn_residuals)):.2f} % Avg.  / {np.abs(nn_residuals).max():.2f} % Max.")

### MIMO Detection Problem

$$
\begin{align}
\min~& \| Ax - b \|^2 \\
\text{s.t.}~& x_j^2 = 1, \quad j = 1, 2, ..., n
\end{align}
$$
Here $x \in \{-1, 1\}^n$ is a signal being transmitted over $m$ noisy channels. The channel matrix is $A$ and the received signal is $b$. The goal is to recover the original signal $x$.

In [ ]:
data_dir = curr_dir / "training_data" / "MIMO"
data_dir.mkdir(exist_ok=True)
model_dir = curr_dir / "models" / "MIMO"
model_dir.mkdir(exist_ok=True)
results_dir = curr_dir / "results" / "MIMO"
results_dir.mkdir(exist_ok=True)

In [ ]:
np.random.seed(10)

# m receivers, n transmitters
m, n = 4, 2

# Channel magnitudes/phases
magnitudes = np.random.uniform(0.4, 0.94, size=(m, n))
phases = np.random.uniform(0, np.pi, size=(m, n))

# Complex channel matrix
A = np.round(magnitudes * np.exp(1j * phases), 2)

# Get training data (complex signals)
N = 5000 # Number of samples
X_arr = 2 * np.random.randint(0, 2, size=(5000, 2)) - 1
y_arr = (A @ X_arr.T).T # Noiseless signal
sigma = 0.1 # Noise standard deviation
w_arr = sigma * (np.random.randn(N, 4) + 1j * np.random.randn(N, 4)) # Noise
b_arr = y_arr + w_arr # Noisy (received) signal

In [ ]:
# Real-valued conversion
A_real = np.block([
    [np.real(A)],
    [np.imag(A)]
])

Q = A_real.T @ A_real

# # Generate c for each sample
# C_list = []
# for k in range(N):
#     b_real = np.concatenate([np.real(b[k]), np.imag(b[k])])
#     c_k = - (A_real.T @ b_real)
#     C_list.append(c_k)
# C_arr = np.array(C_list)

b_real_list = []
for k in range(N):
    b_real = np.concatenate([np.real(b_arr[k]), np.imag(b_arr[k])])
    b_real_list.append(b_real)
b_real_arr = np.array(b_real_list)

In [ ]:
def M_func(b_real):
    c = - (A_real.T @ b_real)
    M = np.block([
        [Q, c.reshape(-1, 1)],
        [c.reshape(1, -1), b_real.T @ b_real]
    ])
    return M

b = b_real_arr[5]

In [ ]:
# SDP Relaxation

# Variables
X = cp.Variable((n+1, n+1), symmetric=True)

# Parameters
M = cp.Parameter((n+1, n+1), symmetric=True)
M.value = M_func(b)

# Objective
objective = cp.Minimize(cp.trace(M @ X))

# Constraints
constraints = [X >> 0]
for j in range(n+1):
    constraints += [X[j,j] == 1]

# Problem
prob = cp.Problem(objective, constraints)
result = prob.solve()

print("Global Objective Function Lower Bound:")
print(result)

# Check rank
tol = 1e-6
eigvals, eigvecs = np.linalg.eigh(X.value)
rank = np.sum(eigvals > tol)
assert rank == 1.0

# Recover optimal solution
idx = np.argmax(eigvals)
lambda1 = eigvals[idx]
v = eigvecs[:, idx]
x = np.sqrt(lambda1) * v

print("Globally Optimal Objective Function Value:")
print(result)

print("Globally Optimal Solution:")
print(x)

In [ ]:
def v(b):
    M.value = M_func(b)
    result = prob.solve()
    # Check rank
    tol = 1e-6
    eigvals, eigvecs = np.linalg.eigh(X.value)
    rank = np.sum(eigvals > tol)
    if rank <= 1.0:
        return result
    else:
        return np.nan

In [ ]:
X_cols = ["b1", "b2", "b3", "b4", "b5", "b6", "b7", "b8"]
df_data = pd.DataFrame(columns=X_cols + ["Cost"])
for i in range(N):
    bi = b_real_arr[i,:]
    df_data.loc[i, X_cols] = bi
    df_data.loc[i, "Cost"] = v(bi)
df_data.to_csv(data_dir / f"training_MIMO_{2 * m}d_{N}samples.csv")

In [ ]:
# NN

# Get data
X, y = df_data[X_cols].values.astype(np.float64), df_data["Cost"].values.astype(np.float64).reshape(-1,1)

# Create training/testing data sets
X_train, y_train, X_test, y_test = create_train_test_datasets(X, y)
train_loader1 = create_batches(X_train, y_train, batch_size=250)
train_loader2 = create_batches(X_train, y_train, batch_size=50)
test_loader = create_batches(X_test, y_test, batch_size=100)

# Initialize model
nn_model = FCNN(input_dim=2 * m)

# Train model

nn_model, nn_train_history_phase2, nn_test_history_phase2 = train_model(
    nn_model,
    train_loader2,
    test_loader,
    optimizer="Adam", 
    n_epochs=3000,
    learning_rate=0.01,
    weight_decay=1e-9,
    scheduler_step_size=250,
    scheduler_gamma=0.9,
    max_norm=100.0
)
nn_train_history = nn_train_history_phase2
nn_test_history = nn_test_history_phase2

nn_save_path = model_dir / f"nn_MIMO_{n}d_{N}samples.pth"
SAVE = True
if SAVE:
    save_model(nn_model, nn_save_path)

In [ ]:
# -----------------------------
# Plotting
# -----------------------------

plt.title("NN Train / Test Error")
plt.plot(np.sqrt(np.array(nn_train_history))[10:], label="Train")
plt.plot(np.sqrt(np.array(nn_test_history))[10:], label="Test")
plt.legend()
#plt.yscale('log')
#plt.axis([0, None, 0, 10])
plt.show()

In [ ]:
# Create test data tensor
test_loader = create_batches(X_test, y_test, batch_size=1, shuffle=False)

# Get NN, ICNN predictions
print("Getting NN predictions...")
nn_pred_arr = pred(nn_model, test_loader)
print("Done.")

# Save results
df_results = pd.DataFrame(columns=X_cols + ["Cost", "NN Pred"])
df_results[X_cols] = X_test
df_results["Cost"] = y_test
df_results["NN Pred"] = nn_pred_arr

df_results.to_csv(results_dir / f"results_MIMO_v2_{n}d_{N}samples.csv")

# Calculate residuals
cost_arr = y_test.flatten()
nn_residuals = (nn_pred_arr - cost_arr) / cost_arr

print(f"NN Error: {100 * np.mean(np.abs(nn_residuals)):.2f} % Avg.  / {100 * np.abs(nn_residuals).max():.2f} % Max.")

In [ ]:
np.mean(np.abs(nn_residuals))

In [ ]:
np.std(cost_arr) / np.mean(cost_arr)